In [ ]:
# CELL 1: GOOGLE COLAB SETUP & DRIVE MOUNT (OPTIONAL)
from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# CELL 2: CONFIGURATION & FILE PATHS
# Provide your Google Colab CSV paths below:
TRAIN_PATTERNS_PATH = "/content/PASTE_STORED_PATTERNS_HERE.csv"

# List of multiple noisy test pattern files to evaluate:
TEST_PATTERNS_PATHS = [
    "/content/PASTE_NOISY_TEST_1_HERE.csv",
    "/content/PASTE_NOISY_TEST_2_HERE.csv"
]

HOPFIELD_MODEL_PATH = "/content/hopfield_weights.npz"


In [ ]:
# CELL 3: IMPORTS & HELPER FUNCTIONS
import numpy as np
import matplotlib.pyplot as plt
import os

def load_patterns_from_csv(file_path):
    """
    Reads pattern rows from CSV. Converts 0/1 binary values into
    bipolar -1/+1 values.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    patterns = []
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            delimiter = "\t" if "\t" in line else ","
            tokens = line.split(delimiter)
            row = [float(t.strip()) for t in tokens]
            patterns.append(row)

    arr = np.array(patterns, dtype=np.float64)
    # Convert binary {0, 1} to bipolar {-1, 1}
    arr[arr == 0] = -1.0
    return arr

def calculate_energy(W, state):
    """Calculates Lyapunov energy: E = -0.5 * (s^T * W * s)"""
    return -0.5 * np.dot(state, np.dot(W, state))

def render_pattern_ascii(pattern_1d, width=5):
    """Visualizes a 1D pattern vector as a 2D ASCII grid."""
    height = len(pattern_1d) // width
    grid_str = ""
    for r in range(height):
        row_slice = pattern_1d[r * width : (r + 1) * width]
        grid_str += " ".join(["#" if val > 0 else "." for val in row_slice]) + "\n"
    return grid_str


In [ ]:
# CELL 4: MODULE 1 - TRAINING MODULE (HEBBIAN WEIGHT LEARNING)
def train_hopfield():
    print("=" * 60)
    print("MODULE 1: TRAINING HOPFIELD NETWORK (WEIGHT COMPUTATION)")
    print("=" * 60)

    # 1. Read Stored Prototype Patterns
    print(f"Reading prototype patterns from: {TRAIN_PATTERNS_PATH}")
    stored_patterns = load_patterns_from_csv(TRAIN_PATTERNS_PATH)
    num_patterns, num_neurons = stored_patterns.shape

    print(f"Loaded {num_patterns} prototype pattern(s), each with {num_neurons} neurons.")

    # 2. Hebbian Learning: W = (1/N) * sum_p (x_p * x_p^T)
    W = np.zeros((num_neurons, num_neurons), dtype=np.float64)
    for p in range(num_patterns):
        vec = stored_patterns[p]
        W += np.outer(vec, vec)

    W = W / num_neurons

    # 3. Hopfield Condition: Zero Diagonal (No self-feedback)
    np.fill_diagonal(W, 0.0)

    # Capacity check: Capacity ~ 0.14 * N
    max_capacity = 0.14 * num_neurons
    print(f"Network Capacity Estimate: Up to ~{max_capacity:.1f} patterns safely recallable.")
    if num_patterns > max_capacity:
        print("Warning: Stored patterns exceed 0.14*N; spurious states may arise.")

    # 4. Save Trained Weight Matrix to File
    np.savez(
        HOPFIELD_MODEL_PATH,
        W=W,
        stored_patterns=stored_patterns,
        num_neurons=num_neurons
    )
    print(f"Hopfield weight matrix saved successfully to: {HOPFIELD_MODEL_PATH}")

# Run Training
train_hopfield()


In [ ]:
# CELL 5: MODULE 2 - TESTING MODULE (PATTERN RETRIEVAL / RECALL)
def test_hopfield():
    print("=" * 60)
    print("MODULE 2: TESTING HOPFIELD NETWORK (RECALL NOISY PATTERNS)")
    print("=" * 60)

    if not os.path.exists(HOPFIELD_MODEL_PATH):
        raise FileNotFoundError(f"Weight file {HOPFIELD_MODEL_PATH} not found. Please run training cell first.")

    # 1. Load Trained Parameters
    data = np.load(HOPFIELD_MODEL_PATH)
    W = data["W"]
    stored_patterns = data["stored_patterns"]
    num_neurons = int(data["num_neurons"])

    # 2. Iterate Over Multiple Test Files
    for file_idx, test_path in enumerate(TEST_PATTERNS_PATHS, start=1):
        print("\n" + "-" * 55)
        print(f"Testing File {file_idx}: {test_path}")
        print("-" * 55)

        if not os.path.exists(test_path):
            print(f"Warning: File {test_path} not found. Skipping.")
            continue

        test_patterns = load_patterns_from_csv(test_path)
        print(f"Found {len(test_patterns)} corrupted test pattern(s).\n")

        for sample_idx, noisy_input in enumerate(test_patterns):
            state = noisy_input.copy()
            energy_history = [calculate_energy(W, state)]

            # 3. Asynchronous Retrieval Dynamics
            max_iter = 50
            converged = False

            for iteration in range(max_iter):
                prev_state = state.copy()

                # Update neurons asynchronously in randomized order
                order = np.random.permutation(num_neurons)
                for i in order:
                    activation = np.dot(W[i, :], state)
                    state[i] = 1.0 if activation >= 0 else -1.0

                current_energy = calculate_energy(W, state)
                energy_history.append(current_energy)

                # Check convergence
                if np.array_equal(state, prev_state):
                    converged = True
                    break

            # 4. Compare with Stored Prototypes
            best_match_idx = -1
            best_bit_match = -1
            for p_idx, proto in enumerate(stored_patterns):
                matching_bits = np.sum(state == proto)
                if matching_bits > best_bit_match:
                    best_bit_match = matching_bits
                    best_match_idx = p_idx

            bit_recovery_percent = (best_bit_match / num_neurons) * 100.0

            print(f"--- Test Sample {sample_idx + 1} ---")
            print(f"Convergence: {'YES (Stable State)' if converged else 'NO (Max Iter Reached)'}")
            print(f"Best Matched Prototype Pattern: #{best_match_idx + 1}")
            print(f"Bit Recovery Match: {best_bit_match}/{num_neurons} ({bit_recovery_percent:.2f}%)")

            # Display ASCII grid if 25 neurons (5x5)
            if num_neurons == 25:
                print("\n[Input Noisy Pattern]:\n" + render_pattern_ascii(noisy_input, 5))
                print("[Retrieved Pattern]:\n" + render_pattern_ascii(state, 5))
                print("[Original Prototype]:\n" + render_pattern_ascii(stored_patterns[best_match_idx], 5))

            # 5. Plot Iteration vs Energy for the first sample
            if sample_idx == 0:
                plt.figure(figsize=(6, 4))
                plt.plot(range(len(energy_history)), energy_history, marker="o", color="purple")
                plt.xlabel("Iteration")
                plt.ylabel("Lyapunov Energy (E)")
                plt.title(f"Hopfield Network: Energy vs Iteration (Sample 1)")
                plt.grid(True)
                plt.show()

# Run Testing
test_hopfield()


## 3. Important Viva Questions & Answers
1. **Q: What is the maximum storage capacity of a Hopfield Network?**  
   *Ans:* The theoretical capacity is approximately $C \approx 0.14 N$, where $N$ is the number of neurons.
2. **Q: Why must diagonal weights $w_{ii}$ be zero?**  
   *Ans:* Setting $w_{ii} = 0$ prevents self-reinforcing loops, ensuring the network state converges toward an energy minimum without endless oscillation.
3. **Q: What are spurious states?**  
   *Ans:* Spurious states are unintended local energy minima created as mathematical side-effects of Hebbian superposition that do not correspond to any stored training pattern.
4. **Q: Why is asynchronous updating preferred over synchronous updating?**  
   *Ans:* Asynchronous updating strictly guarantees monotonic energy descent ($\Delta E \le 0$). Synchronous updating can cause the state to flip endlessly between two states.
